# Analisis de mantenimiento vehicular para prediccion de fallas de motor

In [1]:
from pathlib import Path
import json

from vehicular_ml.data import load_dataset
from vehicular_ml.eda import run_eda
from vehicular_ml.predict import predict_dataframe
from vehicular_ml.training import TrainingConfig, train_and_save

In [2]:
project_root = Path('..').resolve()
data_path = project_root / 'data' / 'raw' / 'vehicular_mantenimiento_sample.csv'
eda_output = project_root / 'artifacts' / 'eda_notebook'
model_output = project_root / 'artifacts' / 'model_notebook'

df = load_dataset(data_path)
df.head()

,kilometraje,temperatura_motor,rpm,horas_uso,voltaje_bateria,marca,combustible,tipo_aceite,codigo_falla,requiere_mantenimiento
0,28742,94.4,874.0,2096.0,12.00,Suzuki,Gasolina,Mineral,P0420,1
1,172530,96.1,1233.0,1961.0,12.87,Hyundai,GNV,Sint'etico,P0420,1
2,147460,94.7,1625.0,1245.0,12.66,Toyota,Di'esel,Semisint'etico,P0420,1
3,102164,98.4,1749.0,1780.0,13.00,Nissan,Gasolina,Sint'etico,P0420,1
4,100933,110.2,1816.0,1240.0,12.61,Chevrolet,GNV,Mineral,P0420,1


In [3]:
eda_results = run_eda(df=df, output_dir=eda_output)
eda_results

{'histogram': '/home/jose/dev/vehiculos/artifacts/eda_notebook/figura_1_distribucion_kilometraje.png',
 'boxplot': '/home/jose/dev/vehiculos/artifacts/eda_notebook/figura_2_boxplot_temperatura.png',
 'heatmap': '/home/jose/dev/vehiculos/artifacts/eda_notebook/figura_3_mapa_calor_correlacion.png',
 'scatter': '/home/jose/dev/vehiculos/artifacts/eda_notebook/figura_4_scatter_kilometraje_falla.png',
 'report': '/home/jose/dev/vehiculos/artifacts/eda_notebook/reporte_eda.md'}

In [4]:
training_summary = train_and_save(
    df=df,
    output_dir=model_output,
    config=TrainingConfig(test_size=0.2, random_state=42, n_estimators=300),
)
print(json.dumps(training_summary, indent=2, ensure_ascii=False))

{
  "config": {
    "test_size": 0.2,
    "random_state": 42,
    "n_estimators": 300
  },
  "train_rows": 480,
  "test_rows": 120,
  "target_distribution": {
    "0": 146,
    "1": 454
  },
  "metrics": {
    "accuracy": 0.9583333333333334,
    "precision": 0.9479166666666666,
    "recall": 1.0,
    "f1": 0.9732620320855615,
    "roc_auc": 0.960212201591512
  },
  "artifacts": {
    "model": "/home/jose/dev/vehiculos/artifacts/model_notebook/model.joblib",
    "metrics": "/home/jose/dev/vehiculos/artifacts/model_notebook/metrics.json",
    "test_predictions": "/home/jose/dev/vehiculos/artifacts/model_notebook/test_predictions.csv"
  }
}


In [5]:
preds = predict_dataframe(model_output / 'model.joblib', df.head(20))
preds.head()

,kilometraje,temperatura_motor,rpm,horas_uso,voltaje_bateria,marca,combustible,tipo_aceite,codigo_falla,prediccion_mantenimiento,probabilidad_mantenimiento
0,28742,94.4,874.0,2096.0,12.00,Suzuki,Gasolina,Mineral,P0420,1,0.936667
1,172530,96.1,1233.0,1961.0,12.87,Hyundai,GNV,Sint'etico,P0420,1,0.880000
2,147460,94.7,1625.0,1245.0,12.66,Toyota,Di'esel,Semisint'etico,P0420,1,0.983333
3,102164,98.4,1749.0,1780.0,13.00,Nissan,Gasolina,Sint'etico,P0420,1,0.960000
4,100933,110.2,1816.0,1240.0,12.61,Chevrolet,GNV,Mineral,P0420,1,0.993333
